Retrieving useful passages and generating a supported answer are different tasks. Week 4 measured whether the system could find evidence. This week asks whether that evidence survives context construction, whether the answer actually uses it, and whether the API communicates success and failure honestly.

The implementation reference is the [Week 5 release](https://github.com/jamwithai/production-agentic-rag-course/tree/week5.0), especially `src/routers/ask.py`, `src/services/ollama/`, `src/schemas/api/ask.py`, and the Gradio interface. The learning goal is to trace an answer from its request through selected chunks, prompt, generation, streaming protocol, and source references.

## RAG as a sequence of evidence-preserving transformations

An answer can fail even when retrieval succeeds. The relevant passage may be truncated, displaced by duplicate chunks, attached to the wrong source ID, or ignored by the generator. Conversely, a plausible answer can be unsupported by the corpus even when it happens to be true.

Three distinct questions help locate the failure: did retrieval find the necessary evidence, did the context include it, and did the generated claims follow from it? A single end-to-end score conceals which intervention is needed. Adding a larger model cannot restore evidence dropped before the model sees the prompt.

For a simplified diagnostic model, let $E$ mean the context contains sufficient evidence and $A$ mean the answer is correctly supported. Then

$$
P(A)=P(A\mid E)P(E)+P(A\mid \neg E)P(\neg E).
$$

An evidence-grounded system aims to avoid unsupported answers when $E$ is false, usually by abstaining or asking for clarification. This decomposition is conceptual; it does not provide measurable probabilities without labeled examples. It shows why evidence coverage and generation quality must be evaluated separately.

The source's standard endpoint performs retrieval, constructs a prompt, calls Ollama, and returns an answer with sources. Its no-chunks branch defines whether the system admits missing evidence or invites the model to improvise.

## Context assembly is a constrained selection problem

A model context window must accommodate instructions, the question, retrieved text, formatting, and generated output. If the total token capacity is $B$, a useful planning inequality is

$$
B_{\mathrm{instructions}}+B_{\mathrm{question}}+B_{\mathrm{evidence}}+B_{\mathrm{output}}\leq B.
$$

The output allowance cannot be spent on retrieved passages without changing generation behavior. The source prompt builder uses compact source metadata and passage text; that reduces overhead but does not, by itself, enforce a tokenizer-aware budget.

Rank order is one selection signal. Diversity, redundancy, evidence completeness, and permissions also matter. Three overlapping chunks from the same paragraph can consume the budget while omitting a second paper needed for comparison. Deduplication by passage overlap or content identity can free space for independent evidence. A blanket one-chunk-per-paper rule can still discard complementary sections needed for the same question.

**Experiment: distinguish passage count from evidence coverage.** The token costs below are stipulated measurements for a toy example. The selector is a simple heuristic: it avoids selecting the same evidence group twice. It is not a general optimizer.


In [ ]:
candidates = [
    {"id": "A1", "evidence_group": "method", "tokens": 70},
    {"id": "A2", "evidence_group": "method", "tokens": 70},
    {"id": "B1", "evidence_group": "limitation", "tokens": 70},
]

def pack(candidates, budget, avoid_duplicates):
    selected, groups = [], set()
    for item in candidates:
        if avoid_duplicates and item["evidence_group"] in groups:
            continue
        if item["tokens"] <= budget:
            selected.append(item)
            groups.add(item["evidence_group"])
            budget -= item["tokens"]
    return selected

for deduplicate in (False, True):
    chosen = pack(candidates, 140, deduplicate)
    print("Deduplicate:", deduplicate, "chunks:", [c["id"] for c in chosen],
          "evidence:", sorted({c["evidence_group"] for c in chosen}))
assert {c["evidence_group"] for c in pack(candidates, 140, True)} == {"method", "limitation"}


Reference output from isolated Python execution:

```text
Deduplicate: False chunks: ['A1', 'A2'] evidence: ['method']
Deduplicate: True chunks: ['A1', 'B1'] evidence: ['limitation', 'method']
```

Both selections use two chunks and the same budget. Only one covers both evidence groups. This is why `chunks_used=2` is not a quality metric. On the full system, measure token counts with the actual tokenizer and inspect the selected text, not just a passage count.

Context ordering also deserves an experiment. [Lost in the Middle](https://arxiv.org/abs/2307.03172) found position-sensitive performance on particular long-context tasks and models. That motivates changing evidence position as a controlled test; it does not prove a universal ordering rule for today's models. A comparable replication needs the model version, context length, prompt, and answer-support judgment.

## A citation is a claim about provenance

The release attaches arXiv references to chunks and asks the generator to cite them. Source IDs should be assembled by the application from retrieved records, not invented by the model. Even a source that really was retrieved may fail to support the sentence citing it.

Two properties require separate evaluation: **citation validity**, meaning the reference points to supplied evidence, and **claim support**, meaning the evidence justifies the associated statement. A citation-set check can test validity; it cannot establish entailment. Support may require human judgment or a separately validated evaluator.

The source prepares a source list using cleaned arXiv IDs. Removing a version suffix makes a convenient link but can point to a later PDF than the one ingested. A reproducible answer depends on a request record containing version-specific provenance, chunk identity, and prompt/model versions. A source list should describe the evidence actually used, not merely all papers returned by an earlier stage.

Retrieved content is also untrusted input. A PDF can contain instructions aimed at changing model behavior. [OWASP's prompt-injection guidance](https://genai.owasp.org/llmrisk/llm01-prompt-injection/) distinguishes indirect instructions arriving through external content. Delimiters and system instructions can help describe the intended boundary, but they do not make that boundary an authorization control. The application should enforce tool permissions, source eligibility, and output handling independently.

## Streaming is a protocol with partial failure

The standard endpoint returns a complete JSON response. The streaming endpoint yields source metadata, text chunks, and a terminal completion object. Earlier visible text improves time to first useful output; it does not automatically reduce total generation work.

Ollama's [streaming API](https://docs.ollama.com/api/streaming) uses newline-delimited JSON. The course API wraps output in Server-Sent Events, with `data:` fields separated by blank lines. These are different framing protocols. A network read can end in the middle of a JSON value, a UTF-8 character, or an event delimiter. A received byte chunk therefore does not necessarily correspond to one event.

**Experiment: parse across arbitrary transport boundaries.** This parser intentionally handles only the course's single-line `data:` events with LF delimiters. It demonstrates buffering and incremental UTF-8 decoding. A general SSE client must implement the full [event-stream specification](https://html.spec.whatwg.org/multipage/server-sent-events.html#parsing-an-event-stream), including other line endings and multiline data.


In [ ]:
import codecs
import json

wire = ('data: {"chunk":"café"}\n\n'
        'data: {"answer":"café","done":true}\n\n').encode("utf-8")

def parse_course_events(parts):
    decoder = codecs.getincrementaldecoder("utf-8")()
    buffer, events = "", []
    for part in parts:
        buffer += decoder.decode(part)
        while "\n\n" in buffer:
            event, buffer = buffer.split("\n\n", 1)
            if not event.startswith("data: "):
                raise ValueError("Unexpected event format")
            events.append(json.loads(event[6:]))
    buffer += decoder.decode(b"", final=True)
    if buffer:
        raise ValueError("Incomplete event stream")
    return events

expected = parse_course_events([wire])
for split in range(len(wire) + 1):
    assert parse_course_events([wire[:split], wire[split:]]) == expected
assert expected[-1]["done"] is True
print("Every two-part byte split parsed correctly:", len(wire) + 1, "cases")
print(expected)


Reference output from isolated Python execution:

```text
Every two-part byte split parsed correctly: 64 cases
[{'chunk': 'café'}, {'answer': 'café', 'done': True}]
```

The assertion tests every two-part boundary, including a boundary inside the accented character. It verifies framing in this restricted example, not all SSE behavior. A connection ending without the terminal object must be treated as incomplete even if some text was displayed. Once headers and partial output have been sent, a server cannot simply replace the stream with a conventional JSON error response.

A complete client contract covers error events, disconnection, duplicate terminal data, cancellation, and backpressure. Retrying a partially consumed answer may launch another generation and duplicate text. A request ID and explicit completion state help distinguish a resumed presentation from a new computation.

## Async syntax does not eliminate blocking

The synchronous OpenSearch call inside an async route and the model client's transport behavior determine where request processing can block. A blocking call can hold an event-loop worker while it waits, reducing concurrency. FastAPI's worker count, thread pools, HTTP connection limits, and model-server capacity then interact; adding workers can increase contention rather than throughput.

Python's [asyncio documentation](https://docs.python.org/3/library/asyncio-task.html#asyncio.to_thread) describes offloading suitable blocking work to a thread. That is one implementation option, not a complete resource policy. The system still needs bounded concurrency, per-stage timeouts, a total request deadline, and cancellation that releases resources.

## Docker lab: trace one answer end to end

1. Use `week5.0`, project name `rag-week5`, real embeddings, and the full Compose stack. Populate its own volumes through ingestion; data does not transfer automatically from earlier project names. Install a small local model inside Ollama after checking available memory.

    ```bash
    docker compose --project-name rag-week5 up --build -d
    docker compose --project-name rag-week5 exec ollama ollama pull llama3.2:1b
    docker compose --project-name rag-week5 exec airflow \
      airflow dags trigger arxiv_paper_ingestion
    curl --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:8000/api/v1/ask \
      -d '{"query":"How do the indexed papers use retrieval?","top_k":3,"use_hybrid":true,"model":"llama3.2:1b"}'
    curl --no-buffer --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:8000/api/v1/stream \
      -d '{"query":"How do the indexed papers use retrieval?","top_k":3,"use_hybrid":true,"model":"llama3.2:1b"}'
    ```

2. Use a known-answer question derived from your actual corpus. Retain the retrieved chunks, final prompt, model identifier, answer, references, time to first content, and total duration. Compare each substantive claim against the supplied passages. Do not judge success from fluent output or a nonempty source list alone.

3. Try an unanswerable question and a corpus record containing an instruction-like sentence in a disposable test corpus. The expected behavior is an explicit evidence limitation and preservation of the application's authority boundaries. Neither outcome is guaranteed by the source; collect counterexamples and use them to improve the design.

4. Interrupt a stream after receiving the first chunk and inspect API and Ollama logs. Does generation stop or continue consuming resources? Then compare a cold model load with repeated warm calls and record concurrency. Performance claims from the upstream notebook are hypotheses to remeasure, not measurements of your laptop.

5. The Gradio launcher is a separate interface process in this release. When ready, launch it with the release's documented host environment or deliberately package it as another container. Do not assume it is included just because the backend Compose stack is running. First validate the HTTP contract directly; the UI should make the same success, incomplete, and error states understandable.

## Engineering review: test the boundaries of a supported answer

1. Take one failed answer and classify its cause: corpus omission, parsing damage, retrieval miss, context selection, unsupported generation, or citation mismatch. Propose the smallest experiment that can distinguish your hypothesis from another cause. For example, supplying the correct passage directly tests generation more cleanly than changing retrieval and the model simultaneously.

2. Define an answer contract with `supported`, `insufficient_evidence`, and `dependency_error` outcomes. Specify which outcomes can be retried and which require a new question or corpus update. Finally, explain why reducing prompt size could improve latency while harming answer completeness. A useful optimization report contains both measurements.

## Sources and scope

- [Week 5 source](https://github.com/jamwithai/production-agentic-rag-course/tree/week5.0): retrieval preparation, prompt construction, standard and streaming APIs, and Gradio.
- [Ollama streaming](https://docs.ollama.com/api/streaming) and the [HTML event-stream specification](https://html.spec.whatwg.org/multipage/server-sent-events.html#parsing-an-event-stream): transport framing.
- [Liu et al., Lost in the Middle](https://arxiv.org/abs/2307.03172): context-position experiments with stated model and task scope.
- [OWASP prompt injection](https://genai.owasp.org/llmrisk/llm01-prompt-injection/): indirect instructions from retrieved content.
- [Python asyncio tasks](https://docs.python.org/3/library/asyncio-task.html): blocking calls, threads, and cancellation behavior.

The deterministic experiments verify selection and framing properties. Answer quality, cancellation, latency, and injection resistance remain integration measurements to collect in Docker.
